<a href="https://colab.research.google.com/github/NSJayaweera/NCD-Risk_Prediction/blob/Chronic_Kidney_Diseases/CKD_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# INSTALL
!pip install streamlit pyngrok -q

In [2]:
# LOAD MODELS
import joblib
import json

SAVE_DIR = "/content/NCD-Risk_Prediction/"

# Load models — joblib (not pickle)
stacking_model = joblib.load(SAVE_DIR + "stacking_model.pkl")
bagging_model  = joblib.load(SAVE_DIR + "bagging_model.pkl")

with open(SAVE_DIR + "feature_order.json", "r") as f:
    feature_order = json.load(f)

print(" Stacking model loaded")
print(" Bagging model loaded")
print(f" Features: {feature_order}")

 Stacking model loaded
 Bagging model loaded
 Features: ['age', 'gender', 'bmi', 'bp_systolic', 'bp_diastolic', 'serum_creatinine', 'blood_urea_nitrogen', 'urine_albumin', 'urine_creatinine', 'albumin_creatinine_ratio', 'albumin_serum', 'uric_acid', 'diabetes_diagnosed', 'bun_creatinine_ratio']


In [3]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import json
import math

# ── Page config ──────────────────────────────────────────────
st.set_page_config(
    page_title = "CKD Risk Prediction",
    page_icon  = "🫀",
    layout     = "wide"
)

# ── Load models ───────────────────────────────────────────────
@st.cache_resource
def load_models():
    SAVE_DIR = "/content/NCD-Risk_Prediction/"
    stacking = joblib.load(SAVE_DIR + "stacking_model.pkl")
    bagging  = joblib.load(SAVE_DIR + "bagging_model.pkl")
    with open(SAVE_DIR + "feature_order.json", "r") as f:
        feature_order = json.load(f)
    return stacking, bagging, feature_order

stacking_model, bagging_model, feature_order = load_models()

# ── eGFR Calculator (CKD-EPI 2021) ───────────────────────────
def calculate_egfr(creatinine, age, gender):
    if creatinine <= 0 or age <= 0:
        return None
    kappa      = 0.7  if gender == 0 else 0.9
    alpha      = -0.241 if gender == 0 else -0.302
    sex_factor = 1.012  if gender == 0 else 1.0
    ratio = creatinine / kappa
    if ratio < 1:
        egfr = 142 * (ratio ** alpha) * (0.9938 ** age) * sex_factor
    else:
        egfr = 142 * (ratio ** -1.200) * (0.9938 ** age) * sex_factor
    return round(egfr, 1)

# ── Title ─────────────────────────────────────────────────────
st.title("🫀 Chronic Kidney Disease Risk Prediction")
st.markdown("Enter patient details below to predict CKD risk using AI models.")
st.divider()

# ── Input Form ────────────────────────────────────────────────
col1, col2 = st.columns(2)

with col1:
    st.subheader("📋 Basic Information")
    age    = st.number_input("Age (years)",
                              min_value=18, max_value=120, value=50)
    gender = st.selectbox("Gender", ["Female", "Male"])
    gender_val = 1 if gender == "Male" else 0

    weight = st.number_input("Weight (kg)",
                              min_value=20.0, max_value=300.0, value=70.0)
    height = st.number_input("Height (cm)",
                              min_value=100.0, max_value=250.0, value=170.0)
    bmi    = round(weight / ((height / 100) ** 2), 1)
    st.info(f"📊 Calculated BMI: **{bmi}**")

    bp_systolic  = st.number_input("Systolic BP (mmHg)",
                                    min_value=60, max_value=260, value=120)
    bp_diastolic = st.number_input("Diastolic BP (mmHg)",
                                    min_value=30, max_value=160, value=80)
    diabetes = st.selectbox("Diabetes Diagnosed", ["No", "Yes"])
    diabetes_val = 1 if diabetes == "Yes" else 0

with col2:
    st.subheader("🔬 Laboratory Values")
    serum_creatinine = st.number_input("Serum Creatinine (mg/dL)",
                                        min_value=0.1, max_value=20.0,
                                        value=1.0, step=0.1)
    blood_urea_nitrogen = st.number_input("Blood Urea Nitrogen (mg/dL)",
                                           min_value=1.0, max_value=200.0,
                                           value=15.0, step=0.5)
    urine_albumin = st.number_input("Urine Albumin (mg/L)",
                                     min_value=0.0, max_value=9000.0,
                                     value=10.0, step=1.0)
    urine_creatinine = st.number_input("Urine Creatinine (mg/dL)",
                                        min_value=1.0, max_value=1200.0,
                                        value=100.0, step=1.0)
    albumin_serum = st.number_input("Serum Albumin (g/dL)",
                                     min_value=0.5, max_value=8.0,
                                     value=4.0, step=0.1)
    uric_acid = st.number_input("Uric Acid (mg/dL)",
                                 min_value=0.5, max_value=25.0,
                                 value=5.0, step=0.1)

# ── Auto-calculated values ────────────────────────────────────
st.divider()
st.subheader("⚡ Auto-Calculated Values")

acr = round(urine_albumin / urine_creatinine * 1000, 2) \
      if urine_creatinine > 0 else 0
bun_cr_ratio = round(blood_urea_nitrogen / serum_creatinine, 2) \
               if serum_creatinine > 0 else 0
egfr_val = calculate_egfr(serum_creatinine, age, gender_val)

c1, c2, c3 = st.columns(3)
c1.metric("Albumin-Creatinine Ratio (mg/g)", f"{acr}")
c2.metric("BUN/Creatinine Ratio",            f"{bun_cr_ratio}")
c3.metric("eGFR (CKD-EPI) — display only",  f"{egfr_val}")

# ── Predict ───────────────────────────────────────────────────
st.divider()

if st.button("🔍 Predict CKD Risk", use_container_width=True):

    patient_data = {
        "age"                      : age,
        "gender"                   : gender_val,
        "bmi"                      : bmi,
        "bp_systolic"              : bp_systolic,
        "bp_diastolic"             : bp_diastolic,
        "serum_creatinine"         : serum_creatinine,
        "blood_urea_nitrogen"      : blood_urea_nitrogen,
        "urine_albumin"            : urine_albumin,
        "urine_creatinine"         : urine_creatinine,
        "albumin_creatinine_ratio" : acr,
        "albumin_serum"            : albumin_serum,
        "uric_acid"                : uric_acid,
        "diabetes_diagnosed"       : diabetes_val,
        "bun_creatinine_ratio"     : bun_cr_ratio,
    }

    patient_df = pd.DataFrame([patient_data])

    for col in feature_order:
        if col not in patient_df.columns:
            patient_df[col] = np.nan
    patient_df = patient_df[feature_order]

    stack_prob = stacking_model.predict_proba(patient_df)[0][1]
    stack_pred = stacking_model.predict(patient_df)[0]
    bag_prob   = bagging_model.predict_proba(patient_df)[0][1]
    bag_pred   = bagging_model.predict(patient_df)[0]

    st.subheader("📊 Prediction Results")
    res1, res2 = st.columns(2)

    for col, name, prob, pred in [
        (res1, "Stacking Model", stack_prob, stack_pred),
        (res2, "Bagging Model",  bag_prob,   bag_pred)
    ]:
        with col:
            st.markdown(f"### {name}")
            if pred == 1:
                st.error("⚠️ CKD Detected")
            else:
                st.success("✅ No CKD Detected")
            st.metric("Probability", f"{prob:.1%}")
            st.progress(float(prob))

    st.divider()
    if stack_pred == bag_pred:
        if stack_pred == 1:
            st.error("🔴 Both models agree: **CKD Detected** — "
                     "Clinical evaluation recommended.")
        else:
            st.success("🟢 Both models agree: **No CKD Detected**")
    else:
        st.warning("🟡 Models disagree — "
                   "Further clinical evaluation recommended.")

    st.divider()
    st.caption(
        "⚠️ This tool is for research purposes only and does not "
        "replace professional medical diagnosis. Always consult a "
        "qualified healthcare provider."
    )

Overwriting app.py


In [4]:
# RUN STREAMLIT APP
from pyngrok import ngrok
import subprocess, time

# Start app
proc = subprocess.Popen(["streamlit", "run", "app.py",
                          "--server.port", "8501"])
time.sleep(3)

# Public URL
ngrok.set_auth_token("3AfyUmo3lVO9yOaHUu0y0mtaaLk_6TA1FXQKAVhqURFAq68TR")
public_url = ngrok.connect(8501)
print(f"\n App running at: {public_url}")


 App running at: NgrokTunnel: "https://uncritically-touchy-solange.ngrok-free.dev" -> "http://localhost:8501"
